# Lab 5, Day 1 - Exploration and Cleaning

Explore, profile, and clean the Titanic dataset, then split it and check your column
groups - no model yet. See `Lab5_Day1_Instructions.md` for the full walkthrough.

This notebook is just a shell: it gives you a place to write and document your work,
but the profiling, the decisions, and the reasoning are yours.

In [68]:
import pandas as pd
import numpy as np

from data import load_titanic

df, source = load_titanic()
print('source:', source)
df.head()


Loaded real Titanic from OpenML  (1309, 14)
source: openml


,Pclass,Survived,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


## Step 1: Load and profile

In [69]:
# Profile the dataset
df.info()
display(df.describe(include='all').T)

# Missingness percentage per column, sorted descending
missing_pct = df.isna().mean() * 100
missing_pct = missing_pct.sort_values(ascending=False)
display(missing_pct)


<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   Pclass     1309 non-null   int64   
 1   Survived   1309 non-null   int64   
 2   Name       1309 non-null   str     
 3   Sex        1309 non-null   category
 4   Age        1046 non-null   float64 
 5   SibSp      1309 non-null   int64   
 6   Parch      1309 non-null   int64   
 7   Ticket     1309 non-null   str     
 8   Fare       1308 non-null   float64 
 9   Cabin      295 non-null    str     
 10  Embarked   1307 non-null   category
 11  boat       486 non-null    str     
 12  body       121 non-null    float64 
 13  home.dest  745 non-null    str     
dtypes: category(2), float64(3), int64(4), str(5)
memory usage: 125.4 KB


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Pclass,1309.0,NaN,NaN,NaN,2.294882,0.837836,1.0,2.0,3.0,3.0,3.0
Survived,1309.0,NaN,NaN,NaN,0.381971,0.486055,0.0,0.0,0.0,1.0,1.0
Name,1309,1307,"Connolly, Miss. Kate",2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sex,1309,2,male,843,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,1046.0,NaN,NaN,NaN,29.881135,14.4135,0.1667,21.0,28.0,39.0,80.0
SibSp,1309.0,NaN,NaN,NaN,0.498854,1.041658,0.0,0.0,0.0,1.0,8.0
Parch,1309.0,NaN,NaN,NaN,0.385027,0.86556,0.0,0.0,0.0,0.0,9.0
Ticket,1309,929,CA. 2343,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Fare,1308.0,NaN,NaN,NaN,33.295479,51.758668,0.0,7.8958,14.4542,31.275,512.3292
Cabin,295,186,C23 C25 C27,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN


body         90.756303
Cabin        77.463713
boat         62.872422
home.dest    43.086325
Age          20.091673
Embarked      0.152788
Fare          0.076394
Pclass        0.000000
Survived      0.000000
Name          0.000000
Sex           0.000000
SibSp         0.000000
Parch         0.000000
Ticket        0.000000
dtype: float64

## Leakage and identifier check

`boat` and `body` directly encode outcome (lifeboat number / recovery body id) - dropping to avoid target leakage. `Name`, `Ticket`, `home.dest` are identifiers / free text with very high cardinality and low signal, dropping for this lab.

## Step 2: The most skipped checks - does missingness predict the target?

In [70]:
# Compare survival rates for Age and Cabin missingness
for col in ["Age", "Cabin"]:
    print(f"\n{col}")
    print("Missing proportion:")
    print(df[col].isna().mean().round(2))
    print("Survival rates:")
    print(df.groupby(df[col].isna())["Survived"].mean().round(2))

df['Age_missing'] = df['Age'].isna().astype(int)
df['Cabin_missing'] = df['Cabin'].isna().astype(int)



Age
Missing proportion:
0.2
Survival rates:
Age
False    0.41
True     0.28
Name: Survived, dtype: float64

Cabin
Missing proportion:
0.77
Survival rates:
Cabin
False    0.65
True     0.30
Name: Survived, dtype: float64


## Observations

- The `Age` column is missing for about 20% of passengers. Passengers with a recorded age have a survival rate of 41%, while passengers with a missing age have a survival rate of only 28%. This suggests that age missingness is informative, so I will create an `Age_missing` indicator feature and include it in my feature set.

- The `Cabin` column is missing for about 77% of passengers. Passengers with cabin information have a survival rate of 65%, compared to only 30% for passengers with missing cabin information. This large difference suggests that cabin missingness is highly informative. Therefore, I will create a `Cabin_missing` indicator feature and include it in my feature set.

## Step 3: Look for impossible values and placeholders

In [71]:
# Check for impossible values and placeholders
# Use string+object to avoid Pandas4Warning (pandas 3 string migration)
for col in df.select_dtypes(include=['string','object']):
    print(f'\n{col}:')
    print(df[col].unique()[:10])

# Check for zero or negative fares and absurd ages
print(f"\nFare non-positive values: {(df['Fare'] <= 0).sum()}")
print(f'Fare zeros: {(df["Fare"] == 0).sum()}, NaN: {df["Fare"].isna().sum()}')
print(f"\nAges outside 0-100: {len(df['Age'][(df['Age'] < 0) | (df['Age'] > 100)])}")

# Fare varies strongly by Pclass and Embarked - justify group median
print("\nMedian Fare by Pclass:")
print(df.groupby("Pclass")["Fare"].median())
print("\nMedian Fare by Embarked:")
print(df.groupby("Embarked", observed=True)["Fare"].median())
print("\nMedian Fare by (Pclass, Embarked):")
print(df.groupby(["Pclass", "Embarked"], observed=True)["Fare"].median())

# Cabin sparsity - why not include in Fare grouping
print(f"\nCabin missing: {df['Cabin'].isna().mean():.2%} (too sparse for grouping)")
print("Cabin present by Pclass:")
print(pd.crosstab(df['Pclass'], df['Cabin'].notna()))



Name:
<StringArray>
[                  'Allen, Miss. Elisabeth Walton',
                  'Allison, Master. Hudson Trevor',
                    'Allison, Miss. Helen Loraine',
            'Allison, Mr. Hudson Joshua Creighton',
 'Allison, Mrs. Hudson J C (Bessie Waldo Daniels)',
                             'Anderson, Mr. Harry',
               'Andrews, Miss. Kornelia Theodosia',
                          'Andrews, Mr. Thomas Jr',
   'Appleton, Mrs. Edward Dale (Charlotte Lamson)',
                         'Artagaveytia, Mr. Ramon']
Length: 10, dtype: str

Ticket:
<StringArray>
[   '24160',   '113781',    '19952',    '13502',   '112050',    '11769',
 'PC 17609', 'PC 17757', 'PC 17477',    '19877']
Length: 10, dtype: str

Cabin:
<StringArray>
['B5', 'C22 C26', 'E12', 'D7', 'A36', 'C101', nan, 'C62 C64', 'B35', 'A23']
Length: 10, dtype: str

boat:
<StringArray>
['2', '11', nan, '3', '10', 'D', '4', '9', '6', 'B']
Length: 10, dtype: str

home.dest:
<StringArray>
[                   'St 

## Step 4: Decide, and write it down

For each column with a problem, add a markdown cell (or a row in a table here) saying
what you did and why. Code with no commentary earns much less credit than the same
code with one sentence of justification.

**Cleaning decisions - what I saw, what I did, and why:**

## 1. Age - missing actually tells you something
- **Saw:** 263 missing (~20%), no ages below 0 or above 100. When age was missing, survival was 28% vs 41% when it was recorded.
- **Did:** Added an `Age_missing` flag (1 if missing, 0 otherwise) and kept the `Age` column to be median-imputed later in the pipeline.
- **Why:** Missingness itself predicts survival, so I didn’t want to just fill it and lose that signal.

## 2. Cabin - super sparse, but useful as a flag
- **Saw:** 77.46% missing (1014/1309). Almost all cabins belong to 1st class (256 of 323 in Pclass 1 have a cabin, but only 23 of 277 in Pclass 2 and 16 of 709 in Pclass 3).
- **Did:** Made a `Cabin_missing` flag and dropped the raw `Cabin` text.
- **Why:** The actual cabin number is too sparse to use, especially if I’d try to group by `(Pclass, Embarked, Cabin)` - I’d get tiny groups for almost no gain. The flag captures the wealth/class signal, and Cabin is heavily tied to Pclass anyway.

## 3. Embarked - barely missing
- **Saw:** Only 2 missing (0.15%), no weird placeholder strings in the uniques.
- **Did:** Will fill with the mode `S` (Southampton, most common) in the pipeline.
- **Why:** Two rows isn’t worth a special flag; mode keeps it simple.

## 4. Fare - zeros are not real fares
- **Saw:** 17 zeros (1.3%) + 1 NaN, no negatives. Fare really depends on class and port - Pclass medians are 60.0 / 15.05 / 8.05, Embarked C 28.5 / Q 7.75 / S 13.0, and the combo matters (e.g. 1/C 76.7 vs 3/Q 7.75).
- **Did:** Treated both zeros and NaN as missing and will impute with the **median Fare for that `(Pclass, Embarked)` group**, fitted on train only (falls back to global median ~14.46 if a group is empty). Used the simple `mask(isna | ==0)` + `groupby.transform` idea.
- **Why:** A free-ticket zero isn’t a real fare, and a global median would really under-value 1st class. The group median keeps the class/port structure.

## 5. Name & Ticket - basically IDs
- **Saw:** Name: 1307 unique out of 1309, Ticket: 929 unique - first 10 values were all different person/ticket codes (`Allen, Miss. Elisabeth Walton`, `24160`, `113781`, `PC 17609`...).
- **Did:** Dropped both.
- **Why:** Way too many distinct values to be useful as categories.

## 6. home.dest - messy and half-missing
- **Saw:** ~43% missing, values like `St Louis, MO`, `Montreal, PQ / Chesterville, ON`, `New York, NY` - free text.
- **Did:** Dropped.
- **Why:** High missing + high cardinality, not reliable for Day 1 baseline.

## 7. boat & body - leakage
- **Saw:** boat - values like `2, 11, 3, 10, D, 4, 9, B` and ~63% missing; body - corpse ID, ~91% missing.
- **Did:** Dropped both.
- **Why:** You only know the lifeboat number or body ID *after* the outcome, so they would leak the target.

## 8. Pclass - numbers but actually categories
- **Saw:** Stored as 1/2/3 ints, but it’s an ordered class label.
- **Did:** Kept it, but moved it to categorical in Step 6.
- **Why:** 3 isn’t ‘more’ than 1 in a numeric sense - one-hot/ordinal encoding makes more sense.


## Step 5: Split first, before fitting anything (`pipeline.py`)

In [72]:
# Build X and y
y = df['Survived']
X = df.drop(columns=['Survived', 'Name', 'Ticket', 'Cabin', 'boat', 'body', 'home.dest'])

# Train-test split (before any fitting!)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y_train distribution: {y_train.value_counts().to_dict()}')
print(f'y_test distribution: {y_test.value_counts().to_dict()}')

X_train shape: (1047, 9)
X_test shape: (262, 9)
y_train distribution: {0: 647, 1: 400}
y_test distribution: {0: 162, 1: 100}


In [ ]:
# Fare: NaN and 0 are both missing - imputed with median per (Pclass, Embarked)
# Simplified code with mask + groupby transform; fit on train only (no leakage)
import numpy as np

# Cabin not used: 77% missing would make (Pclass,Embarked,Cabin) sparse and is collinear with Pclass
# (kept as Cabin_missing indicator only)

# Fit medians on train only - treat 0 as missing via s[s>0]
group_medians = X_train.groupby(['Pclass','Embarked'], observed=True)['Fare'].apply(lambda s: s[s>0].median())
global_median = X_train.loc[X_train['Fare']>0, 'Fare'].median()
print("Group medians (train):")
print(group_medians)
print(f"global median (Fare>0): {global_median:.2f}")

def impute_fare(df, medians=group_medians, glob=global_median):
    """Simplified: mask NaN|0 and map group median."""
    df = df.copy()
    # per-group median via transform (using train medians, fallback to global)
    fill = df.apply(lambda r: medians.loc[(r['Pclass'], r['Embarked'])] if (r['Pclass'], r['Embarked']) in medians.index else glob, axis=1)
    df['Fare'] = df['Fare'].mask(df['Fare'].isna() | (df['Fare']==0), fill)
    # fallback if group median was NaN (empty group)
    df['Fare'] = df['Fare'].fillna(glob)
    return df

#  works when imputing within the same frame (no train/test split)
# df["Fare"] = df["Fare"].mask(
#     df["Fare"].isna() | (df["Fare"]==0),
#     df.groupby(["Pclass","Embarked"])["Fare"].transform(lambda s: s[s>0].median())
# )
# For Day1 we use the train-fit version above to avoid leakage.

print(f"\nBefore: train zeros={(X_train['Fare']==0).sum()} NaN={X_train['Fare'].isna().sum()} | test zeros={(X_test['Fare']==0).sum()} NaN={X_test['Fare'].isna().sum()}")
Xtr_imp = impute_fare(X_train)
Xte_imp = impute_fare(X_test)
print(f"After:  train zeros={(Xtr_imp['Fare']==0).sum()} NaN={Xtr_imp['Fare'].isna().sum()} | test zeros={(Xte_imp['Fare']==0).sum()} NaN={Xte_imp['Fare'].isna().sum()}")


Group medians (train):
Pclass  Embarked
1       C           78.2667
        Q           90.0000
        S           53.1000
2       C           21.3854
        Q           12.3500
        S           18.7500
3       C            7.8958
        Q            7.7500
        S            8.0500
Name: Fare, dtype: float64
global median (Fare>0): 14.46

Before: train zeros=12 NaN=1 | test zeros=5 NaN=0
After:  train zeros=0 NaN=0 | test zeros=0 NaN=0


## Step 6: Column groups - and check them by eye

In [74]:
# Split X's columns into numeric vs categorical by dtype
numeric_cols = X.select_dtypes(include='number').columns.tolist()
categorical_cols = X.select_dtypes(exclude='number').columns.tolist()

# Check if Pclass (ordinal) ended up in numeric - discuss
print('Numeric cols:', numeric_cols)
print('Categorical cols:', categorical_cols)

# Pclass is stored as int but is really an ordinal category (1, 2, 3),
# not a continuous measurement, so we'll move it to categorical.
if 'Pclass' in numeric_cols:
    numeric_cols.remove('Pclass')
    categorical_cols.append('Pclass')
    print('Moved Pclass to categorical (ordinal, not continuous)')


Numeric cols: ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Age_missing', 'Cabin_missing']
Categorical cols: ['Sex', 'Embarked']
Moved Pclass to categorical (ordinal, not continuous)


In [75]:
import joblib
joblib.dump({"X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test}, "split.joblib")
print('Split saved to split.joblib')

Split saved to split.joblib


---
**Before you close this notebook today:**
- Save your split so Day 2 resumes rather than re-derives it - e.g.
  `joblib.dump({"X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test}, "split.joblib")`.
  Re-splitting tomorrow with a different `random_state` would silently invalidate every
  comparison you make against today's work.
- Confirm your split used `stratify=y` and a fixed `random_state`.
- Make sure Step 4's cleaning decisions are actually written down while the reasoning
  is fresh - reconstructing it tomorrow produces visibly thinner justifications.
- Keep this notebook and folder as-is. Day 2 is a new notebook here, not a restart.